In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import perlin_noise
import torch

TODO fits on full curve or on simulated curve from parameters

Phenology curve function

In [ ]:
def make_pheno_curve(
    greenup_start_doy: float,
    senesce_start_doy: float,
    greenup_rate: float,
    senesce_rate: float,
    green_min: float,
    green_max: float,
):
    t = np.linspace(0, 365, num=365)
    green_delta = green_max - green_min
    greenup_end_doy = greenup_start_doy + (green_delta / greenup_rate)
    senesce_end_doy = senesce_start_doy + (green_delta / senesce_rate)
    return np.piecewise(
        t,
        [
            t < greenup_start_doy, # pre green-up
            (greenup_start_doy <= t) & (t < greenup_end_doy), # greenup
            (greenup_end_doy <= t) & (t < senesce_start_doy), # max green
            (senesce_start_doy <= t) & (t < senesce_end_doy), # senescence
            (senesce_end_doy <= t) # post senescence
        ],
        [
            lambda x: green_min,
            lambda x: green_min + greenup_rate * (x - greenup_start_doy),
            lambda x: green_max,
            lambda x: green_max - senesce_rate * (x - senesce_start_doy),
            lambda x: green_min
        ]
    )

In [ ]:
def double_logistic(x, slope1, slope2, beta1, beta2, omega1, omega2):
    return (
        (omega1 / (1 + np.exp(-slope1 * (x - beta1)))) * 
        ((1 - omega2) / (1 + np.exp(slope2 * (x - beta2)))) 
    )

In [ ]:
x = np.linspace(0, 365, num=365)

plt.plot(
    double_logistic(x, 0.1, 0.1, 60, 250, 1, 0)
)

In [ ]:
pheno_endmember_parameters = np.array([
    [0.02, 0.02, 100, 200, 0.50, 0.00],
    [0.10, 0.02,  50, 250, 0.80, 0.00],
    [0.05, 0.08, 150, 250, 1.00, 0.00]
])

x = np.linspace(0, 365, num=365)

pheno_endmembers = np.array([
    double_logistic(x, *params) for params in pheno_endmember_parameters
])

In [ ]:
plt.plot(pheno_endmembers.T)
plt.show()

In [ ]:
noise_gen = perlin_noise.PerlinNoise(octaves=8, seed=42)
X, Y, Z = np.meshgrid(np.linspace(0, 1, 100), np.linspace(0, 1, 100), np.linspace(0, 1, pheno_endmember_parameters.shape[0]))

@np.vectorize
def noise_gen_vector(x, y, z):
    return noise_gen((x, y, z))

surface = noise_gen_vector(X, Y, Z)

In [ ]:
surface = surface + 0.5
surface = surface / surface.sum(axis=-1, keepdims=True)
assert np.allclose(surface.sum(axis=-1), 1)

In [ ]:
fig, axes = plt.subplots(1, 3)

for i, ax in enumerate(axes.flat):
    ax.imshow(surface[:, :, i], vmin=0, vmax=1)
    ax.set_xticks([])
    ax.set_yticks([])

plt.show()

Combine endmember and cover fraction to get observed phenology

In [ ]:
pheno_obs = (surface.reshape(-1, surface.shape[-1]) @ pheno_endmembers).reshape(surface.shape[0], surface.shape[1], -1)

In [ ]:
plt.plot(pheno_obs[50, 10])

Create a torch module to estimate endmember parameters.

In [ ]:
class PhenoMixture(torch.nn.Module):
    def __init__(self, n_endmembers, *args, **kwargs):
        super(PhenoMixture, self).__init__(*args, **kwargs)
        self.n_endmembers = n_endmembers
        
        self.beta1 = torch.nn.Parameter(torch.ones(n_endmembers) * 0.33)
        self.beta2 = torch.nn.Parameter(torch.ones(n_endmembers) * 0.66)
        
        self.slope1 = torch.nn.Parameter(torch.ones(n_endmembers) * 0.1)
        self.slope2 = torch.nn.Parameter(torch.ones(n_endmembers) * 0.1)
        
        self.omega1 = torch.nn.Parameter(torch.ones(n_endmembers))
        self.omega2 = torch.nn.Parameter(torch.zeros(n_endmembers))
    
    @staticmethod
    def double_logistic(x: torch.tensor, slope1, slope2, beta1, beta2, omega1, omega2):
        return (
            (omega1 / (1 + torch.exp(-slope1 * (x - beta1)))) * 
            ((1 - omega2) / (1 + torch.exp(slope2 * (x - beta2)))) 
        )
    
    def forward(self, x: torch.tensor, endmember_cover: torch.tensor) -> torch.tensor:
        ret = torch.zeros(x.shape)
        for i in range(self.n_endmembers):
            ret += self.double_logistic(
                x, 
                self.slope1[i], self.slope2[i], 
                self.beta1[i], self.beta2[i], 
                self.omega1[i], self.omega2[i]
            ) * endmember_cover[i]

        return ret
        

In [ ]:
mymodel = PhenoMixture(3)

In [ ]:
x = torch.linspace(0, 1, 100)
cover = torch.ones(3) / 3

mixture = mymodel(x, cover)

In [ ]:
plt.plot(mixture.detach().numpy())